# LAPANet · Inference Walkthrough

This notebook walks through the complete inference pipeline of **LAPANet**, a
*Local All-Pass Attention Network* for cardiac motion estimation from
sub-sampled k-space.

Given a complex-valued 5D k-space array of shape `(F, S, C, H, W)`
— frames × slices × coils × height × width — and two selected frames
`(z1, t1)` and `(z2, t2)`, we:

1. Load the k-space from `.npy` / `.npz` / `.h5` / `.mat`
2. Render the coil-combined magnitude images of both inputs (`ifft2c → sum over coils → abs`)
3. Apply training-time preprocessing (pad/crop → normalize → `fft2c`)
4. Run the network forward pass
5. Visualize the estimated motion field as **color encoding** and a **quiver plot**

In [ ]:
import sys
import os
from pathlib import Path

# Adjust this to your local LAPANet checkout if you're not running inside it.
PROJECT_ROOT = Path.cwd()
while PROJECT_ROOT.name != "LAPANet" and PROJECT_ROOT.parent != PROJECT_ROOT:
    PROJECT_ROOT = PROJECT_ROOT.parent

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

print("Project root:", PROJECT_ROOT)
print("Python:", sys.version.split()[0])

In [ ]:
import numpy as np
import torch
import h5py
import matplotlib
import matplotlib.pyplot as plt
from PIL import Image

from pathlib import Path
from argparse import Namespace

# LAPANet internals
from src.models.LAPANet import LAPANet2D
from src.utils import ifft2c, fft2c, crop, zpad

# Optional: flow color encoding
try:
    from flow_vis import flow_to_color
    HAS_FLOW_VIS = True
    print("flow_vis available")
except ImportError:
    HAS_FLOW_VIS = False
    print("flow_vis NOT available — color tab will be skipped")

# Nicer inline figures
%matplotlib inline
plt.rcParams["figure.dpi"] = 110
plt.rcParams["figure.facecolor"] = "black"
plt.rcParams["axes.facecolor"] = "black"
plt.rcParams["savefig.facecolor"] = "black"
plt.rcParams["text.color"] = "white"
plt.rcParams["axes.labelcolor"] = "white"
plt.rcParams["xtick.color"] = "white"
plt.rcParams["ytick.color"] = "white"

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", device)

## 1 · Configuration

Set the path to your k-space file and the indices of the two frames you want to
compare. The canonical layout is `(Frames, Slices, Coils, H, W)`.

For cardiac motion, pick **adjacent frames in the same slice**:
`z1 = z2`, `t2 = t1 + 1`.

In [ ]:
# ---- Input file ----------------------------------------------------------
KSPACE_PATH = "/path/to/cine_sax.mat"   #  .mat

# ---- Frame pair ----------------------------------------------------------
Z1, T1 = 0, 0        # fixed input  (slice, frame)
Z2, T2 = 0, 1        # moving input (slice, frame)

# ---- Model ---------------------------------------------------------------
OUT_SHAPE   = (512, 512)
NUM_COILS   = 10
CHECKPOINT  = None
# Change None to your checkpoint path or it will be pulled from the HF Hub:
HF_REPO_ID         = "AyaGhoul/LAPANet"
HF_CHECKPOINT_NAME = "model_best.pt"

# ---- Quiver rendering ----------------------------------------------------
QUIVER_GRID  = 20
QUIVER_SCALE = 1.0

print("File        :", KSPACE_PATH)
print("Pair        :", f"(z1={Z1}, t1={T1})  →  (z2={Z2}, t2={T2})")
print("Out shape   :", OUT_SHAPE)
print("Checkpoint  :", CHECKPOINT or f"{HF_REPO_ID}/{HF_CHECKPOINT_NAME}")

## 2 · Load the k-space file

Supported formats and the way each is handled:

| Extension       | Reader    | Notes |
|-----------------|-----------|-------|
| `.npy`          | `np.load` | complex array |
| `.npz`          | `np.load` | picks `kspace` or the first key |
| `.h5` / `.hdf5` | `h5py`    | any dataset whose name matches a k-space candidate |
| `.mat`          | `h5py`    | MATLAB v7.3 (HDF5) compound `real`/`imag` dtype |

MATLAB stores arrays column-major, so the last two axes are swapped on read
to give conventional `(…, H, W)` ordering.

In [ ]:
KSPACE_CANDIDATES = [
    "kspace", "kspace_full", "kspace_sub04", "kspace_sub08",
    "kspace_sub10", "kspace_sub4", "kspace_sub8", "full_kspace",
]


def _find_h5_key(h5file, candidates):
    """Case-insensitive lookup with prefix / substring matching."""
    all_keys = []
    try:
        def _visit(name, obj):
            if isinstance(obj, h5py.Dataset):
                all_keys.append(name)
        h5file.visititems(_visit)
    except Exception:
        all_keys = list(h5file.keys())

    if not all_keys:
        raise KeyError("HDF5 file has no readable datasets")

    lowered = {k.lower(): k for k in all_keys}
    for cand in candidates:
        c = cand.lower()
        if c in lowered:
            return lowered[c]
        for lk, real in lowered.items():
            if lk.startswith(c) or c in lk:
                return real
    raise KeyError(f"None of {candidates} found. Available: {all_keys}")


def _read_h5_key(h5file, key):
    """Read HDF5 dataset, following groups/refs, fixing MATLAB compound + column-major."""
    node = h5file[key]
    if isinstance(node, h5py.Group):
        node = node["data"] if "data" in node else node[list(node.keys())[0]]

    data = node[()]
    if isinstance(data, h5py.Reference):
        data = h5file[data][()]

    # MATLAB v7.3 complex: compound dtype with 'real'/'imag' fields
    if data.dtype.names is not None and {"real", "imag"} <= set(data.dtype.names):
        data = (data["real"] + 1j * data["imag"]).astype(np.complex64)

    # MATLAB column-major → swap last two axes
    if hasattr(data, "ndim") and data.ndim >= 2:
        data = np.swapaxes(data, -1, -2)

    return data


def load_kspace(path):
    """Load complex k-space from .npy/.npz/.h5/.hdf5/.mat."""
    path = Path(path)
    ext = path.suffix.lower()

    if ext == ".npy":
        kspace = np.load(path)

    elif ext == ".npz":
        data = np.load(path)
        key = "kspace" if "kspace" in data else list(data.keys())[0]
        kspace = data[key]

    elif ext in (".h5", ".hdf5", ".mat"):
        with h5py.File(path, "r") as f:
            key = _find_h5_key(f, KSPACE_CANDIDATES)
            kspace = _read_h5_key(f, key)

    else:
        raise ValueError(f"Unsupported file format: {ext}")

    return kspace.astype(np.complex64)

In [ ]:
kspace = load_kspace(KSPACE_PATH)

print("Shape :", kspace.shape)
print("Dtype :", kspace.dtype)

# Enforce 5D canonical layout
if kspace.ndim != 5:
    raise ValueError(
        f"Expected 5D (F, S, C, H, W) k-space; got ndim={kspace.ndim} "
        f"with shape {kspace.shape}. Reshape before proceeding."
    )

F, S, C, H, W = kspace.shape
print(f"Frames={F} · Slices={S} · Coils={C} · H={H} · W={W}")

if C != NUM_COILS:
    print(f"⚠  Model expects {NUM_COILS} coils, got {C}.")

## 3 · Extract the two frames

`(z1, t1)` selects the **fixed** frame and `(z2, t2)` the **moving** frame.
Each is a `(Coils, H, W)` complex k-space slab.

In [ ]:
def _check(idx, size, name):
    if not (0 <= idx < size):
        raise ValueError(f"{name}={idx} out of range for axis of size {size}")

_check(T1, F, "t1"); _check(T2, F, "t2")
_check(Z1, S, "z1"); _check(Z2, S, "z2")

ksp_fix = kspace[T1, Z1]   # (C, H, W)
ksp_mov = kspace[T2, Z2]   # (C, H, W)

print("Fixed  k-space:", ksp_fix.shape, ksp_fix.dtype)
print("Moving k-space:", ksp_mov.shape, ksp_mov.dtype)

## 4 · Coil-combined magnitude images

To visualize the input, we invert the k-space along the two spatial axes:

$$
m(x, y) = \left| \sum_{c=1}^{C} \text{IFFT}_{2D}\big( k_c \big)(x, y) \right|
$$

and normalize to `[0, 1]` for display.

In [ ]:
def kspace_to_magnitude(ksp_2d, normalize=True):
    """(C, H, W) complex k-space → (H, W) coil-combined magnitude image."""
    img = ifft2c(ksp_2d, axes=[-2, -1])         # (C, H, W) complex
    combined = np.abs(np.sum(img, axis=0))      # sum over coils → abs
    if normalize:
        mx = combined.max()
        if mx > 0:
            combined = combined / mx
    return combined


mag_fix = kspace_to_magnitude(ksp_fix)
mag_mov = kspace_to_magnitude(ksp_mov)

fig, axes = plt.subplots(1, 2, figsize=(10, 5))
axes[0].imshow(mag_fix, cmap="gray")
axes[0].set_title(f"Fixed  · z1={Z1}, t1={T1}")
axes[0].axis("off")
axes[1].imshow(mag_mov, cmap="gray")
axes[1].set_title(f"Moving · z2={Z2}, t2={T2}")
axes[1].axis("off")
plt.tight_layout()
plt.show()

## 5 · Preprocess for the model

Training used the following normalization on every k-space slab:

1. `ifft2c` → coil images
2. **pad** or **crop** to the training spatial size (`OUT_SHAPE`)
3. divide by the **maximum magnitude** (global normalization)
4. `fft2c` → normalized k-space

This is applied *identically* at inference time so the model sees the same
distribution it was trained on.

In [ ]:
def preprocess_kspace(ksp_2d, out_shape=OUT_SHAPE):
    img = ifft2c(ksp_2d, axes=[-2, -1])
    target = (ksp_2d.shape[0], *out_shape)
    if img.shape[-2] < out_shape[-2] or img.shape[-1] < out_shape[-1]:
        img = zpad(img, target)
    if img.shape[-2] > out_shape[-2] or img.shape[-1] > out_shape[-1]:
        img = crop(img, target)
    mx = np.abs(img).max()
    if mx == 0:
        mx = 1.0
    return fft2c(img / mx, axes=[-2, -1])


ksp_fix_pp = preprocess_kspace(ksp_fix)
ksp_mov_pp = preprocess_kspace(ksp_mov)

# Add batch dim → (1, C, H, W) complex tensors
k_fix = torch.from_numpy(ksp_fix_pp).unsqueeze(0).to(device)
k_mov = torch.from_numpy(ksp_mov_pp).unsqueeze(0).to(device)

print("k_fix:", k_fix.shape, k_fix.dtype)
print("k_mov:", k_mov.shape, k_mov.dtype)

## 6 · Load LAPANet2D

The architecture is a U-Net-shaped all-pass attention network with
attention heads `[1] × 9` and encoder/decoder filter widths
`[16, 32, 64, 192, 384, 192, 64, 32, 16]`.

In [ ]:
model_args = Namespace(
    att_heads=[1, 1, 1, 1, 1, 1, 1, 1, 1],
    global_filters=[4, 16, 32, 128],
    encoder_decoder_filters=[16, 32, 64, 192, 384, 192, 64, 32, 16],
    deep_supervision=False,
    return_translation=True,
    shift_input=True,
    num_coils=NUM_COILS,
)

model = LAPANet2D(model_args).to(device)
print("Model instantiated.")
print("Filters :", model_args.encoder_decoder_filters)
print("Heads   :", model_args.att_heads)

In [ ]:
if CHECKPOINT:
    ckpt_path = CHECKPOINT
else:
    from huggingface_hub import hf_hub_download
    ckpt_path = hf_hub_download(repo_id=HF_REPO_ID, filename=HF_CHECKPOINT_NAME)

checkpoint = torch.load(ckpt_path, map_location=device, weights_only=False)
state_dict = checkpoint.get("model_state_dict", checkpoint)
model.load_state_dict(state_dict)
model.eval()

print("Loaded weights from:", ckpt_path)

## 7 · Forward pass

`LAPANet2D.forward(k_fix, k_mov)` returns a tuple. We take the first element,
the estimated flow field, and drop the translation head.

In [ ]:
with torch.inference_mode():
    out = model(k_fix, k_mov)

print("Model output type:", type(out))
if isinstance(out, tuple):
    print("Tuple length    :", len(out))
    for i, o in enumerate(out):
        print(f"  out[{i}]: {tuple(o.shape) if torch.is_tensor(o) else type(o)}")

flow_tensor = out[0] if isinstance(out, tuple) else out
print("Flow tensor:", tuple(flow_tensor.shape), flow_tensor.dtype)

## 8 · Convert the flow tensor to `(u, v)`

The model may return the flow in any of these layouts:

| Shape           | Interpretation    |
|-----------------|-------------------|
| `(1, 2, H, W)`  | channels-first    |
| `(2, H, W)`     | channels-first, squeezed |
| `(1, H, W, 2)`  | channels-last     |
| `(H, W, 2)`     | channels-last, squeezed |

We normalize everything to two `(H, W)` float32 arrays.

In [ ]:
def flow_tensor_to_uv(tensor):
    arr = tensor.detach().cpu().numpy()
    arr = np.squeeze(arr)

    if arr.ndim == 3 and arr.shape[0] == 2:
        u, v = arr[0], arr[1]
    elif arr.ndim == 3 and arr.shape[-1] == 2:
        u, v = arr[..., 0], arr[..., 1]
    elif arr.ndim == 2:
        u, v = arr, np.zeros_like(arr)
    else:
        raise ValueError(f"Cannot interpret flow tensor shape {arr.shape}")

    return u.astype(np.float32), v.astype(np.float32)


u, v = flow_tensor_to_uv(flow_tensor)
print("u:", u.shape, "range", float(u.min()), "→", float(u.max()))
print("v:", v.shape, "range", float(v.min()), "→", float(v.max()))

## 9 · Visualize as color

The optical-flow convention: **hue = direction**, **saturation/brightness = magnitude**.

We use [`flow_vis.flow_to_color`](https://github.com/tomrunia/OpticalFlow_Visualization)
which implements the standard Middlebury color wheel.

In [ ]:
if HAS_FLOW_VIS:
    flow = np.stack([u, v], axis=-1).astype(np.float32)
    flow_rgb = flow_to_color(flow, convert_to_bgr=False)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(flow_rgb)
    ax.set_title(f"Flow · color  ·  z1={Z1}, t1={T1}  →  z2={Z2}, t2={T2}")
    ax.axis("off")
    plt.tight_layout()
    plt.show()
else:
    print("flow_vis not installed — skipping color rendering.")
    print("Install with:  pip install flow_vis")

## 10 · Visualize as quiver

A quiver plot samples `(u, v)` at every `grid_size` pixels and draws an arrow
per sample on a black background. `v` is negated because image coordinates
put `y` downward while the plot axes put it upward.

In [ ]:
def plot_flow_quiver(u, v, grid_size=20, scale=1.0, title=""):
    h, w = u.shape
    y_grid, x_grid = np.mgrid[0:h:grid_size, 0:w:grid_size]
    u_s = u[::grid_size, ::grid_size]
    v_s = v[::grid_size, ::grid_size]

    fig, ax = plt.subplots(figsize=(7, 7), dpi=110)
    fig.patch.set_facecolor("black")
    ax.set_facecolor("black")

    ax.imshow(np.zeros((h, w, 3)), cmap="gray")

    ax.quiver(
        x_grid, y_grid,
        u_s, -v_s,
        angles="xy", scale_units="xy", scale=scale,
        color="yellow", width=0.003,
        headwidth=3, headlength=4, alpha=0.8,
    )
    ax.set_title(title, color="white")
    ax.axis("off")
    ax.invert_yaxis()
    ax.grid(True, alpha=0.3, color="white", linestyle="--", linewidth=0.5)
    plt.tight_layout()
    plt.show()


plot_flow_quiver(
    u, v,
    grid_size=QUIVER_GRID,
    scale=QUIVER_SCALE,
    title=f"Flow · quiver  ·  z1={Z1}, t1={T1}  →  z2={Z2}, t2={T2}",
)

## 11 · Sanity checks

A quick summary of the flow field helps catch trivial or degenerate outputs
(e.g. all-zero flow, saturated flow, NaNs).

In [ ]:
def flow_summary(u, v, name="flow"):
    mag = np.sqrt(u**2 + v**2)
    ang = np.arctan2(v, u)
    print(f"--- {name} ---")
    print(f"  u        : [{u.min():+.3f}, {u.max():+.3f}]  mean={u.mean():+.3f}")
    print(f"  v        : [{v.min():+.3f}, {v.max():+.3f}]  mean={v.mean():+.3f}")
    print(f"  |flow|   : [{mag.min():.3f}, {mag.max():.3f}]  mean={mag.mean():.3f}")
    print(f"  angle    : [{np.degrees(ang.min()):+.1f}°, {np.degrees(ang.max()):+.1f}°]")
    print(f"  NaN u/v  : {np.isnan(u).any()} / {np.isnan(v).any()}")


flow_summary(u, v)

## 13 · Loop over multiple frames

To reconstruct motion for a whole cine series, iterate over adjacent frame
pairs in a fixed slice and collect the flow fields.

In [ ]:
SLICE = 0
flows = []

for t in range(F - 1):
    ksp_a = kspace[t,     SLICE]
    ksp_b = kspace[t + 1, SLICE]

    ka = torch.from_numpy(preprocess_kspace(ksp_a)).unsqueeze(0).to(device)
    kb = torch.from_numpy(preprocess_kspace(ksp_b)).unsqueeze(0).to(device)

    with torch.inference_mode():
        out_t = model(ka, kb)
    ft = out_t[0] if isinstance(out_t, tuple) else out_t
    u_t, v_t = flow_tensor_to_uv(ft)
    flows.append((u_t, v_t))

print(f"Computed {len(flows)} flow fields for slice {SLICE}.")

# Plot a filmstrip of the color-encoded magnitudes
if HAS_FLOW_VIS and flows:
    n_show = min(len(flows), 6)
    idx = np.linspace(0, len(flows) - 1, n_show).astype(int)

    fig, axes = plt.subplots(1, n_show, figsize=(3 * n_show, 3))
    if n_show == 1:
        axes = [axes]
    for ax, i in zip(axes, idx):
        uu, vv = flows[i]
        rgb = flow_to_color(np.stack([uu, vv], axis=-1).astype(np.float32),
                            convert_to_bgr=False)
        ax.imshow(rgb)
        ax.set_title(f"t={i}→{i+1}", color="white", fontsize=9)
        ax.axis("off")
    plt.tight_layout()
    plt.show()